# Indian Sign Language: Mediapipe Holistic Landmarks Generation

In [8]:
!pip install -q mediapipe opencv-python tensorflow scikit-learn pandas numpy matplotlib seaborn

In [9]:
import tensorflow as tf
import mediapipe as mp
import cv2
import numpy as np

print("TensorFlow:", tf.__version__)
print("MediaPipe:", mp.__version__)
print("OpenCV:", cv2.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.19.0
MediaPipe: 1.1.0
OpenCV: 4.13.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [10]:
import os

for root, dirs, files in os.walk("/kaggle/input"):
    level = root.replace("/kaggle/input", "").count(os.sep)

    if level <= 3:
        print(
            "  " * level +
            os.path.basename(root) +
            f" ({len(files)} files)"
        )

input (0 files)
  datasets (0 files)
    swaptr (0 files)
      isl-video (1 files)


In [11]:
import os

DATASET_PATH = "/kaggle/input/datasets/swaptr/isl-video"

print("Dataset exists:", os.path.exists(DATASET_PATH))
print("Dataset path:", DATASET_PATH)

for item in os.listdir(DATASET_PATH):
    full_path = os.path.join(DATASET_PATH, item)

    if os.path.isdir(full_path):
        print(f"[DIR]  {item}")
    else:
        size_mb = os.path.getsize(full_path) / (1024 * 1024)
        print(f"[FILE] {item} - {size_mb:.2f} MB")

Dataset exists: True
Dataset path: /kaggle/input/datasets/swaptr/isl-video
[DIR]  adjectives
[DIR]  electronics
[DIR]  animals
[DIR]  means_of_transportation
[DIR]  jobs
[DIR]  pronouns
[DIR]  greetings
[DIR]  days_and_time
[DIR]  seasons
[DIR]  places
[DIR]  clothes
[DIR]  society
[DIR]  home
[FILE] label_map.json - 0.00 MB
[DIR]  people
[DIR]  colours


In [12]:
for root, dirs, files in os.walk(DATASET_PATH):
    print("\nPATH:", root)
    print("Directories:", dirs[:10])
    print("Files:", files[:10])
    
    # Don't print everything
    if root.count(os.sep) > DATASET_PATH.count(os.sep) + 2:
        dirs[:] = []


PATH: /kaggle/input/datasets/swaptr/isl-video
Directories: ['adjectives', 'electronics', 'animals', 'means_of_transportation', 'jobs', 'pronouns', 'greetings', 'days_and_time', 'seasons', 'places']
Files: ['label_map.json']

PATH: /kaggle/input/datasets/swaptr/isl-video/adjectives
Directories: ['female', 'nice', 'cheap', 'expensive', 'tall', 'curved', 'warm', 'hard', 'rich', 'alive']
Files: []

PATH: /kaggle/input/datasets/swaptr/isl-video/adjectives/female
Directories: []
Files: ['MVI_9767.MOV', 'MVI_9768.MOV', 'MVI_9645.MOV', 'MVI_9644.MOV', 'MVI_9770.MOV', 'MVI_9769.MOV', 'MVI_9643.MOV', 'MVI_9642.MOV']

PATH: /kaggle/input/datasets/swaptr/isl-video/adjectives/nice
Directories: []
Files: ['MVI_9590.MOV', 'MVI_9589.MOV', 'MVI_9591.MOV', 'MVI_9588.MOV']

PATH: /kaggle/input/datasets/swaptr/isl-video/adjectives/cheap
Directories: []
Files: ['MVI_9752.MOV', 'MVI_9754.MOV', 'MVI_9620.MOV', 'MVI_9750.MOV', 'MVI_9617.MOV', 'MVI_9619.MOV', 'MVI_9618.MOV', 'MVI_9751.MOV']

PATH: /kaggle/inp

In [13]:
from pathlib import Path
from collections import Counter

DATASET_PATH = Path("/kaggle/input/datasets/swaptr/isl-video")

VIDEO_EXTENSIONS = {".mov", ".mp4", ".avi", ".mkv"}

samples = []

for category in DATASET_PATH.iterdir():
    if not category.is_dir():
        continue

    for label_dir in category.iterdir():
        if not label_dir.is_dir():
            continue

        videos = [
            f for f in label_dir.iterdir()
            if f.suffix.lower() in VIDEO_EXTENSIONS
        ]

        if videos:
            samples.append({
                "category": category.name,
                "label": label_dir.name,
                "count": len(videos)
            })

print("Number of classes:", len(samples))
print("Number of videos:", sum(x["count"] for x in samples))

print("\nFirst 30 classes:")
for x in samples[:30]:
    print(
        f"{x['category']:30s} "
        f"{x['label']:25s} "
        f"{x['count']:4d}"
    )

Number of classes: 262
Number of videos: 4284

First 30 classes:
adjectives                     female                       8
adjectives                     nice                         4
adjectives                     cheap                        8
adjectives                     expensive                    8
adjectives                     tall                        21
adjectives                     curved                       8
adjectives                     warm                        21
adjectives                     hard                         8
adjectives                     rich                         8
adjectives                     alive                        8
adjectives                     good                        21
adjectives                     flat                         8
adjectives                     dry                         21
adjectives                     famous                       8
adjectives                     cool                        21
adjec

In [14]:
counts = Counter()

for category in DATASET_PATH.iterdir():
    if not category.is_dir():
        continue

    for label_dir in category.iterdir():
        if not label_dir.is_dir():
            continue

        count = sum(
            1 for f in label_dir.iterdir()
            if f.suffix.lower() in VIDEO_EXTENSIONS
        )

        if count:
            counts[label_dir.name] = count

print("\nSmallest classes:")
for label, count in sorted(counts.items(), key=lambda x: x[1])[:20]:
    print(label, count)

print("\nLargest classes:")
for label, count in sorted(counts.items(), key=lambda x: x[1], reverse=True)[:20]:
    print(label, count)


Smallest classes:
nice 4
thin 4
female 8
cheap 8
expensive 8
curved 8
hard 8
rich 8
alive 8
flat 8
famous 8
poor 8
deaf 8
sad 8
weak 8
dead 8
low 8
male 8
high 8
strong 8

Largest classes:
bird 25
court 23
short 22
smalllittle 22
goodafternoon 22
bank 22
trainstation 22
park 22
storeorshop 22
location 22
tall 21
warm 21
good 21
dry 21
cool 21
long 21
narrow 21
new 21
loud 21
healthy 21


In [15]:
from pathlib import Path
import pandas as pd

DATASET_PATH = Path("/kaggle/input/datasets/swaptr/isl-video")

VIDEO_EXTENSIONS = {".mov", ".mp4", ".avi", ".mkv"}

records = []

for category in sorted(DATASET_PATH.iterdir()):

    if not category.is_dir():
        continue

    for label_dir in sorted(category.iterdir()):

        if not label_dir.is_dir():
            continue

        for video in sorted(label_dir.iterdir()):

            if video.suffix.lower() in VIDEO_EXTENSIONS:

                records.append({
                    "category": category.name,
                    "label": label_dir.name,
                    "video_path": str(video)
                })

df = pd.DataFrame(records)

print("Videos:", len(df))
print("Classes:", df["label"].nunique())

display(df.head())

Videos: 4284
Classes: 262


,category,label,video_path
0,adjectives,alive,/kaggle/input/datasets/swaptr/isl-video/adject...
1,adjectives,alive,/kaggle/input/datasets/swaptr/isl-video/adject...
2,adjectives,alive,/kaggle/input/datasets/swaptr/isl-video/adject...
3,adjectives,alive,/kaggle/input/datasets/swaptr/isl-video/adject...
4,adjectives,alive,/kaggle/input/datasets/swaptr/isl-video/adject...


In [16]:
print(df["label"].value_counts().describe())

count    262.000000
mean      16.351145
std        4.535807
min        4.000000
25%       14.000000
50%       15.000000
75%       20.000000
max       25.000000
Name: count, dtype: float64


In [18]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    random_state=42,
    stratify=df["label"]
)

print("Train:", len(train_df))
print("Test :", len(test_df))

print("\nMinimum train samples per class:",
      train_df["label"].value_counts().min())

print("Minimum test samples per class:",
      test_df["label"].value_counts().min())

Train: 3427
Test : 857

Minimum train samples per class: 3
Minimum test samples per class: 1


In [19]:
test_video = df.iloc[0]["video_path"]

sequence = video_to_sequence(test_video)

print("Video:", test_video)
print("Sequence shape:", sequence.shape)
print("Data type:", sequence.dtype)
print("Contains NaN:", np.isnan(sequence).any())

NameError: name 'video_to_sequence' is not defined

In [20]:
import cv2
import numpy as np
import mediapipe as mp

# Configuration
SEQUENCE_LENGTH = 40
FEATURES = 258

# MediaPipe
mp_holistic = mp.solutions.holistic


def extract_landmarks(results):

    # -------------------------
    # Pose: 33 × 4 = 132
    # -------------------------
    if results.pose_landmarks:
        pose = np.array([
            [lm.x, lm.y, lm.z, lm.visibility]
            for lm in results.pose_landmarks.landmark
        ], dtype=np.float32).flatten()
    else:
        pose = np.zeros(132, dtype=np.float32)

    # -------------------------
    # Left hand: 21 × 3 = 63
    # -------------------------
    if results.left_hand_landmarks:
        left_hand = np.array([
            [lm.x, lm.y, lm.z]
            for lm in results.left_hand_landmarks.landmark
        ], dtype=np.float32).flatten()
    else:
        left_hand = np.zeros(63, dtype=np.float32)

    # -------------------------
    # Right hand: 21 × 3 = 63
    # -------------------------
    if results.right_hand_landmarks:
        right_hand = np.array([
            [lm.x, lm.y, lm.z]
            for lm in results.right_hand_landmarks.landmark
        ], dtype=np.float32).flatten()
    else:
        right_hand = np.zeros(63, dtype=np.float32)

    return np.concatenate([
        pose,
        left_hand,
        right_hand
    ])


def video_to_sequence(video_path):

    frames = []

    cap = cv2.VideoCapture(video_path)

    if not cap.isOpened():
        print("Could not open:", video_path)
        return None

    with mp_holistic.Holistic(
        static_image_mode=False,
        model_complexity=1,
        smooth_landmarks=True,
        refine_face_landmarks=False,
        min_detection_confidence=0.5,
        min_tracking_confidence=0.5
    ) as holistic:

        while True:

            ret, frame = cap.read()

            if not ret:
                break

            # BGR → RGB
            frame_rgb = cv2.cvtColor(
                frame,
                cv2.COLOR_BGR2RGB
            )

            results = holistic.process(frame_rgb)

            landmarks = extract_landmarks(results)

            frames.append(landmarks)

    cap.release()

    if len(frames) == 0:
        return None

    frames = np.asarray(
        frames,
        dtype=np.float32
    )

    # -------------------------
    # Temporal sampling
    # -------------------------
    if len(frames) >= SEQUENCE_LENGTH:

        indices = np.linspace(
            0,
            len(frames) - 1,
            SEQUENCE_LENGTH
        ).astype(int)

        frames = frames[indices]

    # -------------------------
    # Padding
    # -------------------------
    else:

        padded = np.zeros(
            (SEQUENCE_LENGTH, FEATURES),
            dtype=np.float32
        )

        padded[:len(frames)] = frames

        frames = padded

    return frames


print("Functions loaded successfully.")

AttributeError: module 'mediapipe' has no attribute 'solutions'

In [21]:
import mediapipe as mp

print("MediaPipe version:", mp.__version__)
print("Has solutions:", hasattr(mp, "solutions"))
print("Has tasks:", hasattr(mp, "tasks"))

MediaPipe version: 1.1.0
Has solutions: False
Has tasks: True


In [22]:
import mediapipe as mp

print(dir(mp.tasks))

['BaseOptions', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'audio', 'components', 'decision', 'retrieval', 'text', 'vision']


In [23]:
print(dir(mp.tasks.vision))

['FaceDetector', 'FaceDetectorOptions', 'FaceDetectorResult', 'FaceLandmarker', 'FaceLandmarkerOptions', 'FaceLandmarkerResult', 'FaceLandmarksConnections', 'GestureRecognizer', 'GestureRecognizerOptions', 'GestureRecognizerResult', 'HandLandmarker', 'HandLandmarkerOptions', 'HandLandmarkerResult', 'HandLandmarksConnections', 'HolisticLandmarker', 'HolisticLandmarkerOptions', 'HolisticLandmarkerResult', 'ImageClassifier', 'ImageClassifierOptions', 'ImageClassifierResult', 'ImageEmbedder', 'ImageEmbedderOptions', 'ImageEmbedderResult', 'ImageProcessingOptions', 'ImageSegmenter', 'ImageSegmenterOptions', 'InteractiveSegmenter', 'InteractiveSegmenterBrushMode', 'InteractiveSegmenterLegacy', 'InteractiveSegmenterLegacyOptions', 'InteractiveSegmenterLegacyRegionOfInterest', 'InteractiveSegmenterOptions', 'InteractiveSegmenterStroke', 'InteractiveSegmenterStrokePoint', 'ObjectDetector', 'ObjectDetectorOptions', 'ObjectDetectorResult', 'PoseLandmark', 'PoseLandmarker', 'PoseLandmarkerOptions'

In [24]:
!wget -q -O /kaggle/working/holistic_landmarker.task \
https://storage.googleapis.com/mediapipe-models/holistic_landmarker/holistic_landmarker/1/holistic_landmarker.task

import os
print("Model exists:", os.path.exists("/kaggle/working/holistic_landmarker.task"))
print("Size:", os.path.getsize("/kaggle/working/holistic_landmarker.task") / (1024*1024), "MB")

Model exists: True
Size: 0.0 MB


In [26]:
import os

path = "/kaggle/working/holistic_landmarker.task"

print("Exists:", os.path.exists(path))
print("Bytes:", os.path.getsize(path))

Exists: True
Bytes: 0


In [27]:
!rm -f /kaggle/working/holistic_landmarker.task

In [28]:
import inspect
import mediapipe as mp

print(inspect.signature(mp.tasks.vision.HolisticLandmarkerOptions))

(base_options: mediapipe.tasks.python.core.base_options.BaseOptions, running_mode: mediapipe.tasks.python.vision.core.vision_task_running_mode.VisionTaskRunningMode = <VisionTaskRunningMode.IMAGE: 'IMAGE'>, min_face_detection_confidence: float = 0.5, min_face_suppression_threshold: float = 0.5, min_face_landmarks_confidence: float = 0.5, min_pose_detection_confidence: float = 0.5, min_pose_suppression_threshold: float = 0.5, min_pose_landmarks_confidence: float = 0.5, min_hand_landmarks_confidence: float = 0.5, output_face_blendshapes: bool = False, output_segmentation_mask: bool = False, result_callback: Optional[Callable[[mediapipe.tasks.python.vision.holistic_landmarker.HolisticLandmarkerResult, mediapipe.tasks.python.vision.core.image.Image, int], NoneType]] = None) -> None


In [29]:
print(inspect.signature(mp.tasks.vision.HolisticLandmarker.create_from_options))

(options: mediapipe.tasks.python.vision.holistic_landmarker.HolisticLandmarkerOptions) -> 'HolisticLandmarker'


In [30]:
print(mp.tasks.vision.HolisticLandmarkerOptions.__doc__)

Options for the holistic landmarker task.

  Attributes:
    base_options: Base options for the holistic landmarker task.
    running_mode: The running mode of the task. Default to the image mode.
      HolisticLandmarker has three running modes: 1) The image mode for
      detecting holistic landmarks on single image inputs. 2) The video mode for
      detecting holistic landmarks on the decoded frames of a video. 3) The live
      stream mode for detecting holistic landmarks on the live stream of input
      data, such as from camera. In this mode, the "result_callback" below must
      be specified to receive the detection results asynchronously.
    min_face_detection_confidence: The minimum confidence score for the face
      detection to be considered successful.
    min_face_suppression_threshold: The minimum non-maximum-suppression
      threshold for face detection to be considered overlapped.
    min_face_landmarks_confidence: The minimum confidence score for the face
      l

In [31]:
!rm -f /kaggle/working/holistic_landmarker.task

In [32]:
!wget -O /kaggle/working/holistic_landmarker.task \
https://storage.googleapis.com/mediapipe-models/holistic_landmarker/holistic_landmarker/1/holistic_landmarker.task

--2026-10-08 15:24:59--  https://storage.googleapis.com/mediapipe-models/holistic_landmarker/holistic_landmarker/1/holistic_landmarker.task
Resolving storage.googleapis.com (storage.googleapis.com)... 104.154.125.27, 34.3.1.27, 34.153.3.27, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|104.154.125.27|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-10-08 15:24:59 ERROR 404: Not Found.



In [33]:
!wget -O /kaggle/working/pose_landmarker.task \
https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/1/pose_landmarker_lite.task

--2026-10-08 15:25:20--  https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/1/pose_landmarker_lite.task
Resolving storage.googleapis.com (storage.googleapis.com)... 34.153.2.27, 34.144.171.27, 34.144.170.27, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|34.153.2.27|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-10-08 15:25:20 ERROR 404: Not Found.



In [34]:
from pathlib import Path

matches = []

for root in [
    Path("/kaggle/input"),
    Path("/kaggle/working")
]:
    if root.exists():
        matches.extend(root.rglob("*.task"))

print("Found:", len(matches))

for p in matches[:50]:
    print(p, p.stat().st_size / (1024 * 1024), "MB")

Found: 2
/kaggle/working/holistic_landmarker.task 0.0 MB
/kaggle/working/pose_landmarker.task 0.0 MB


In [35]:
import mediapipe as mp

print(mp.__file__)
print(mp.__version__)

/usr/local/lib/python3.12/dist-packages/mediapipe/__init__.py
1.1.0


In [36]:
import inspect
from mediapipe.tasks.python import vision

print(inspect.getsource(vision.HolisticLandmarkerOptions))

@dataclasses.dataclass
class HolisticLandmarkerOptions:
  """Options for the holistic landmarker task.

  Attributes:
    base_options: Base options for the holistic landmarker task.
    running_mode: The running mode of the task. Default to the image mode.
      HolisticLandmarker has three running modes: 1) The image mode for
      detecting holistic landmarks on single image inputs. 2) The video mode for
      detecting holistic landmarks on the decoded frames of a video. 3) The live
      stream mode for detecting holistic landmarks on the live stream of input
      data, such as from camera. In this mode, the "result_callback" below must
      be specified to receive the detection results asynchronously.
    min_face_detection_confidence: The minimum confidence score for the face
      detection to be considered successful.
    min_face_suppression_threshold: The minimum non-maximum-suppression
      threshold for face detection to be considered overlapped.
    min_face_landmarks_

In [40]:
import os
import glob

for pattern in [
    "/kaggle/input/**/*.task",
    "/kaggle/input/**/*holistic*",
    "/kaggle/input/**/*pose*",
]:
    print("\nPATTERN:", pattern)
    results = glob.glob(pattern, recursive=True)
    for x in results[:30]:
        print(x)


PATTERN: /kaggle/input/**/*.task

PATTERN: /kaggle/input/**/*holistic*

PATTERN: /kaggle/input/**/*pose*


In [41]:
from pathlib import Path

for p in Path("/kaggle/input").iterdir():
    print(p)

/kaggle/input/datasets


In [42]:
from pathlib import Path

for p in Path("/kaggle/input").rglob("*"):
    if p.is_file():
        print(p)

/kaggle/input/datasets/swaptr/isl-video/label_map.json
/kaggle/input/datasets/swaptr/isl-video/adjectives/female/MVI_9767.MOV
/kaggle/input/datasets/swaptr/isl-video/adjectives/female/MVI_9768.MOV
/kaggle/input/datasets/swaptr/isl-video/adjectives/female/MVI_9645.MOV
/kaggle/input/datasets/swaptr/isl-video/adjectives/female/MVI_9644.MOV
/kaggle/input/datasets/swaptr/isl-video/adjectives/female/MVI_9770.MOV
/kaggle/input/datasets/swaptr/isl-video/adjectives/female/MVI_9769.MOV
/kaggle/input/datasets/swaptr/isl-video/adjectives/female/MVI_9643.MOV
/kaggle/input/datasets/swaptr/isl-video/adjectives/female/MVI_9642.MOV
/kaggle/input/datasets/swaptr/isl-video/adjectives/nice/MVI_9590.MOV
/kaggle/input/datasets/swaptr/isl-video/adjectives/nice/MVI_9589.MOV
/kaggle/input/datasets/swaptr/isl-video/adjectives/nice/MVI_9591.MOV
/kaggle/input/datasets/swaptr/isl-video/adjectives/nice/MVI_9588.MOV
/kaggle/input/datasets/swaptr/isl-video/adjectives/cheap/MVI_9752.MOV
/kaggle/input/datasets/swaptr/i

In [43]:
from pathlib import Path
from collections import Counter

files = [
    p for p in Path("/kaggle/input").rglob("*")
    if p.is_file()
]

print("Total files:", len(files))

exts = Counter(p.suffix.lower() for p in files)
print(exts)

Total files: 4285
Counter({'.mov': 3679, '.mp4': 605, '.json': 1})


In [44]:
!rm -f /kaggle/working/holistic_landmarker.task
!rm -f /kaggle/working/pose_landmarker.task

In [45]:
!wget -q -O /kaggle/working/holistic_landmarker.task \
https://storage.googleapis.com/mediapipe-models/holistic_landmarker/holistic_landmarker/float16/latest/holistic_landmarker.task

In [46]:
import os

MODEL_PATH = "/kaggle/working/holistic_landmarker.task"

print("Exists:", os.path.exists(MODEL_PATH))
print("Bytes:", os.path.getsize(MODEL_PATH))
print("MB:", os.path.getsize(MODEL_PATH) / (1024 * 1024))

Exists: True
Bytes: 13683609
MB: 13.04970645904541


In [47]:
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

landmarker = mp_vision.HolisticLandmarker.create_from_options(
    mp_vision.HolisticLandmarkerOptions(
        base_options=mp_python.BaseOptions(
            model_asset_path=MODEL_PATH
        ),
        running_mode=mp_vision.RunningMode.VIDEO,
    )
)

print("✅ Holistic Landmarker loaded!")

✅ Holistic Landmarker loaded!


INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1791473517.631825     262 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791473517.666222     261 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791473517.671307     264 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791473517.672954     261 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791473517.673101     262 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791473517.704763     

In [50]:
label_set = set()
jobs = []

for fp in glob.glob(INPUT_GLOB, recursive=True):
    if not fp.lower().endswith(SUPPORTED_EXT):
        continue

    p = Path(fp)
    label_set.add(p.parent.name)

    out = (
        OUTPUT_ROOT
        / p.parent.parent.name
        / p.parent.name
        / f"{p.stem}.parquet"
    )

    if out.exists():
        continue

    jobs.append((fp, out))

print(f"{len(jobs)} videos to process")
print(f"{len(label_set)} classes")

NameError: name 'INPUT_GLOB' is not defined

In [51]:
t0 = time.time()

with Pool(
    cpu_count(),
    initializer=_init_worker,
    initargs=(MODEL_PATH,)
) as pool:

    for i, _ in enumerate(
        pool.imap_unordered(
            process_one_video,
            jobs,
            chunksize=1
        ),
        1
    ):
        if i % 25 == 0:
            print(f"[{i}/{len(jobs)}] {time.time() - t0:.1f}s")

print(f"done in {time.time() - t0:.1f}s")

NameError: name 'time' is not defined

In [52]:
import glob
import json
import time
from pathlib import Path
from multiprocessing import Pool, cpu_count

import cv2
import numpy as np
import pandas as pd
import mediapipe as mp

from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision


# =========================
# Paths
# =========================

MODEL_PATH = "/kaggle/working/holistic_landmarker.task"

INPUT_GLOB = "/kaggle/input/datasets/swaptr/isl-video/**/*"

OUTPUT_ROOT = Path("/kaggle/working/keypoints")

SUPPORTED_EXT = (".mov", ".mp4")

DOWNSCALE_WIDTH = 480
FRAME_STRIDE = 1


# =========================
# Landmark configuration
# =========================

FACE_COUNT = 478
POSE_COUNT = 33
HAND_COUNT = 21

PER_FRAME_TOTAL = (
    FACE_COUNT
    + HAND_COUNT
    + POSE_COUNT
    + HAND_COUNT
)

TYPE_NAMES = np.array([
    "face",
    "left_hand",
    "pose",
    "right_hand"
])


print("✅ Setup loaded")
print("Model:", MODEL_PATH)
print("Model exists:", Path(MODEL_PATH).exists())
print("Input:", INPUT_GLOB)
print("Output:", OUTPUT_ROOT)

✅ Setup loaded
Model: /kaggle/working/holistic_landmarker.task
Model exists: True
Input: /kaggle/input/datasets/swaptr/isl-video/**/*
Output: /kaggle/working/keypoints


In [53]:
label_set = set()
jobs = []

for fp in glob.glob(INPUT_GLOB, recursive=True):
    if not fp.lower().endswith(SUPPORTED_EXT):
        continue

    p = Path(fp)
    label_set.add(p.parent.name)

    out = (
        OUTPUT_ROOT
        / p.parent.parent.name
        / p.parent.name
        / f"{p.stem}.parquet"
    )

    if out.exists():
        continue

    jobs.append((fp, out))

print(f"Videos to process: {len(jobs)}")
print(f"Classes: {len(label_set)}")

Videos to process: 4284
Classes: 262


In [54]:
t0 = time.time()

with Pool(
    cpu_count(),
    initializer=_init_worker,
    initargs=(MODEL_PATH,)
) as pool:

    for i, _ in enumerate(
        pool.imap_unordered(
            process_one_video,
            jobs,
            chunksize=1
        ),
        1
    ):
        if i % 25 == 0:
            print(
                f"[{i}/{len(jobs)}] "
                f"{time.time() - t0:.1f}s"
            )

print(f"Done in {time.time() - t0:.1f}s")

NameError: name '_init_worker' is not defined

In [55]:
_LANDMARKER = None

def _init_worker(model_path):
    global _LANDMARKER

    _LANDMARKER = mp_vision.HolisticLandmarker.create_from_options(
        mp_vision.HolisticLandmarkerOptions(
            base_options=mp_python.BaseOptions(
                model_asset_path=model_path
            ),
            running_mode=mp_vision.RunningMode.VIDEO,
        )
    )

print("✅ Worker function defined")

✅ Worker function defined


In [56]:
print("_init_worker:", _init_worker)
print("process_one_video:", process_one_video)

_init_worker: <function _init_worker at 0x7a8f78170180>


NameError: name 'process_one_video' is not defined

In [57]:
t0 = time.time()

with Pool(
    cpu_count(),
    initializer=_init_worker,
    initargs=(MODEL_PATH,)
) as pool:

    for i, _ in enumerate(
        pool.imap_unordered(
            process_one_video,
            jobs,
            chunksize=1
        ),
        1
    ):
        if i % 25 == 0:
            print(
                f"[{i}/{len(jobs)}] "
                f"{time.time() - t0:.1f}s"
            )

print(f"Done in {time.time() - t0:.1f}s")

NameError: name 'process_one_video' is not defined

In [58]:
def _write_block(landmarks, type_id, count, base, buf, frame_idx, fy):
    sl = slice(base, base + count)

    if landmarks:
        buf["x"][sl] = np.fromiter(
            (lm.x for lm in landmarks),
            dtype=np.float32,
            count=count
        )

        buf["y"][sl] = np.fromiter(
            (lm.y for lm in landmarks),
            dtype=np.float32,
            count=count
        ) * fy

        buf["z"][sl] = np.fromiter(
            (lm.z for lm in landmarks),
            dtype=np.float32,
            count=count
        )

    else:
        buf["x"][sl] = np.nan
        buf["y"][sl] = np.nan
        buf["z"][sl] = np.nan

    buf["frame"][sl] = frame_idx
    buf["type_id"][sl] = type_id
    buf["landmark_index"][sl] = np.arange(
        count,
        dtype=np.int32
    )


def process_one_video(args):
    src_path, out_path = args

    cap = cv2.VideoCapture(str(src_path))

    n_hint = int(
        cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0
    )

    cap_rows = max(n_hint, 1) * PER_FRAME_TOTAL

    buf = {
        k: np.empty(cap_rows, dtype=t)
        for k, t in [
            ("frame", np.int32),
            ("type_id", np.int8),
            ("landmark_index", np.int32),
            ("x", np.float32),
            ("y", np.float32),
            ("z", np.float32),
        ]
    }

    offset = 0
    frame_no = 0
    raw_idx = 0

    while True:
        ok, image = cap.read()

        if not ok:
            break

        if raw_idx % FRAME_STRIDE != 0:
            raw_idx += 1
            continue

        h, w, _ = image.shape

        if DOWNSCALE_WIDTH and w > DOWNSCALE_WIDTH:
            new_h = int(
                h * DOWNSCALE_WIDTH / w
            )

            image = cv2.resize(
                image,
                (DOWNSCALE_WIDTH, new_h),
                interpolation=cv2.INTER_AREA
            )

            h, w = new_h, DOWNSCALE_WIDTH

        fy = h / w

        mp_img = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=cv2.cvtColor(
                image,
                cv2.COLOR_BGR2RGB
            )
        )

        result = _LANDMARKER.detect_for_video(
            mp_img,
            time.monotonic_ns() // 1_000_000
        )

        face = result.face_landmarks or None
        lh = result.left_hand_landmarks or None
        pose = result.pose_landmarks or None
        rh = result.right_hand_landmarks or None

        _write_block(
            face,
            0,
            FACE_COUNT,
            offset,
            buf,
            frame_no,
            fy
        )
        offset += FACE_COUNT

        _write_block(
            lh,
            1,
            HAND_COUNT,
            offset,
            buf,
            frame_no,
            fy
        )
        offset += HAND_COUNT

        _write_block(
            pose,
            2,
            POSE_COUNT,
            offset,
            buf,
            frame_no,
            fy
        )
        offset += POSE_COUNT

        _write_block(
            rh,
            3,
            HAND_COUNT,
            offset,
            buf,
            frame_no,
            fy
        )
        offset += HAND_COUNT

        frame_no += 1
        raw_idx += 1

    cap.release()

    for k in buf:
        buf[k] = buf[k][:offset]

    type_str = TYPE_NAMES[
        buf["type_id"]
    ]

    row_id = np.char.add(
        np.char.add(
            buf["frame"].astype("U"),
            "-"
        ),
        np.char.add(
            type_str,
            np.char.add(
                "-",
                buf["landmark_index"].astype("U")
            )
        )
    )

    out_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    pd.DataFrame({
        "frame": buf["frame"],
        "row_id": row_id,
        "type": type_str,
        "landmark_index": buf["landmark_index"],
        "x": buf["x"],
        "y": buf["y"],
        "z": buf["z"],
    }).to_parquet(
        out_path,
        index=False
    )

    return str(src_path)


print("✅ process_one_video defined")

✅ process_one_video defined


In [59]:
print("_init_worker:", _init_worker)
print("process_one_video:", process_one_video)

_init_worker: <function _init_worker at 0x7a8f78170180>
process_one_video: <function process_one_video at 0x7a8f78170ea0>


In [60]:
t0 = time.time()

with Pool(
    cpu_count(),
    initializer=_init_worker,
    initargs=(MODEL_PATH,)
) as pool:

    for i, _ in enumerate(
        pool.imap_unordered(
            process_one_video,
            jobs,
            chunksize=1
        ),
        1
    ):
        if i % 25 == 0:
            print(
                f"[{i}/{len(jobs)}] "
                f"{time.time() - t0:.1f}s"
            )

print(f"Done in {time.time() - t0:.1f}s")

W0000 00:00:1791473837.128443     333 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791473837.142546     347 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791473837.143107     344 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791473837.176384     350 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791473837.180821     333 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791473837.194474     334 inference_feedback_manager.cc:121] Feedback manager 

[25/4284] 61.8s
[50/4284] 115.4s
[75/4284] 175.9s
[100/4284] 232.2s
[125/4284] 282.9s
[150/4284] 343.4s
[175/4284] 394.9s
[200/4284] 449.4s
[225/4284] 505.9s
[250/4284] 562.6s
[275/4284] 615.6s
[300/4284] 665.0s
[325/4284] 717.5s
[350/4284] 767.3s
[375/4284] 818.7s
[400/4284] 875.7s
[425/4284] 931.1s
[450/4284] 992.1s
[475/4284] 1058.6s
[500/4284] 1112.1s
[525/4284] 1168.2s
[550/4284] 1218.3s
[575/4284] 1268.4s
[600/4284] 1320.1s
[625/4284] 1379.8s
[650/4284] 1429.3s
[675/4284] 1488.6s
[700/4284] 1543.1s
[725/4284] 1596.7s
[750/4284] 1646.1s
[775/4284] 1700.8s
[800/4284] 1758.9s
[825/4284] 1821.7s
[850/4284] 1878.3s
[875/4284] 1938.9s
[900/4284] 2001.6s
[925/4284] 2058.2s
[950/4284] 2124.0s
[975/4284] 2202.6s
[1000/4284] 2281.5s
[1025/4284] 2364.4s
[1050/4284] 2440.6s
[1075/4284] 2516.0s
[1100/4284] 2586.1s
[1125/4284] 2667.9s
[1150/4284] 2755.3s
[1175/4284] 2849.0s
[1200/4284] 2940.6s
[1225/4284] 3010.7s
[1250/4284] 3079.8s
[1275/4284] 3151.5s
[1300/4284] 3217.4s
[1325/4284] 3282.0s
[

In [61]:
from pathlib import Path
import pandas as pd

KEYPOINT_ROOT = Path("/kaggle/working/keypoints")

files = list(KEYPOINT_ROOT.rglob("*.parquet"))

print("Parquet files:", len(files))

if files:
    sample_file = files[0]
    sample = pd.read_parquet(sample_file)

    print("\nSample file:")
    print(sample_file)

    print("\nShape:", sample.shape)

    print("\nLandmark types:")
    print(sample["type"].value_counts())

    print("\nColumns:")
    print(sample.columns.tolist())

    print("\nFrames:", sample["frame"].nunique())

Parquet files: 4284

Sample file:
/kaggle/working/keypoints/home/gift/MVI_4949.parquet

Shape: (34839, 7)

Landmark types:
type
face          30114
pose           2079
left_hand      1323
right_hand     1323
Name: count, dtype: int64

Columns:
['frame', 'row_id', 'type', 'landmark_index', 'x', 'y', 'z']

Frames: 63


In [62]:
from pathlib import Path
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

KEYPOINT_ROOT = Path("/kaggle/working/keypoints")

SEQUENCE_LENGTH = 40
FEATURES = 225

# Fixed landmark order
LANDMARK_TYPES = ["pose", "left_hand", "right_hand"]
LANDMARK_COUNTS = {
    "pose": 33,
    "left_hand": 21,
    "right_hand": 21
}


def extract_frame_features(df):
    """
    Convert one Parquet file into:
        (num_frames, 225)
    """

    frames = sorted(df["frame"].unique())
    output = []

    for frame_id in frames:
        frame_df = df[df["frame"] == frame_id]

        feature_vector = []

        for landmark_type in LANDMARK_TYPES:
            part = frame_df[frame_df["type"] == landmark_type]

            # Preserve landmark index ordering
            part = part.sort_values("landmark_index")

            expected = LANDMARK_COUNTS[landmark_type]

            xyz = np.zeros((expected, 3), dtype=np.float32)

            for _, row in part.iterrows():
                idx = int(row["landmark_index"])

                if idx < expected:
                    xyz[idx] = [
                        row["x"],
                        row["y"],
                        row["z"]
                    ]

            # Replace NaN/Inf
            xyz = np.nan_to_num(
                xyz,
                nan=0.0,
                posinf=0.0,
                neginf=0.0
            )

            feature_vector.extend(xyz.flatten())

        output.append(feature_vector)

    return np.asarray(output, dtype=np.float32)


def temporal_sample(sequence, target_length=40):
    """
    Convert arbitrary frame count -> exactly 40 frames.
    """

    n_frames = len(sequence)

    if n_frames == target_length:
        return sequence

    if n_frames > target_length:
        # Uniform temporal sampling
        indices = np.linspace(
            0,
            n_frames - 1,
            target_length
        ).astype(np.int32)

        return sequence[indices]

    # Padding for short videos
    padded = np.zeros(
        (target_length, sequence.shape[1]),
        dtype=np.float32
    )

    padded[:n_frames] = sequence

    # Repeat final frame rather than leaving excessive zeros
    if n_frames > 0:
        padded[n_frames:] = sequence[-1]

    return padded


parquet_files = sorted(KEYPOINT_ROOT.rglob("*.parquet"))

print("Parquet files:", len(parquet_files))

X = []
labels = []
metadata = []

for file in tqdm(parquet_files):

    df = pd.read_parquet(file)

    sequence = extract_frame_features(df)

    sequence = temporal_sample(
        sequence,
        SEQUENCE_LENGTH
    )

    assert sequence.shape == (40, 225)

    # Class = parent directory
    label = file.parent.name

    X.append(sequence)
    labels.append(label)

    metadata.append({
        "file": str(file),
        "label": label
    })


X = np.asarray(X, dtype=np.float32)
labels = np.asarray(labels)

print("\n================================")
print("DATASET CREATED")
print("================================")

print("X shape:", X.shape)
print("Labels:", labels.shape)
print("Features/frame:", X.shape[-1])
print("Frames/video:", X.shape[1])
print("Classes:", len(np.unique(labels)))

Parquet files: 4284


  0%|          | 0/4284 [00:00<?, ?it/s]


DATASET CREATED
X shape: (4284, 40, 225)
Labels: (4284,)
Features/frame: 225
Frames/video: 40
Classes: 262


In [63]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import numpy as np
import json

# --------------------------------------------------
# 1. Encode class labels
# --------------------------------------------------

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(labels)

classes = label_encoder.classes_

print("Number of classes:", len(classes))
print("Encoded labels:", y_encoded.shape)

# --------------------------------------------------
# 2. Stratified 80/20 split
# --------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.20,
    random_state=42,
    stratify=y_encoded
)

print("\nTrain:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nTest:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

# --------------------------------------------------
# 3. Calculate normalization ONLY from training data
# --------------------------------------------------

# Mean/std across samples and frames
mean = X_train.mean(axis=(0, 1), keepdims=True)
std = X_train.std(axis=(0, 1), keepdims=True)

# Prevent division by zero
std = np.where(std < 1e-6, 1.0, std)

# Normalize
X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

# --------------------------------------------------
# 4. Save preprocessing information
# --------------------------------------------------

np.save("/kaggle/working/mean.npy", mean)
np.save("/kaggle/working/std.npy", std)

classes_dict = {
    str(i): str(class_name)
    for i, class_name in enumerate(classes)
}

with open("/kaggle/working/classes.json", "w") as f:
    json.dump(classes_dict, f, indent=2)

# --------------------------------------------------
# 5. Save datasets
# --------------------------------------------------

np.save("/kaggle/working/X_train.npy", X_train)
np.save("/kaggle/working/X_test.npy", X_test)
np.save("/kaggle/working/y_train.npy", y_train)
np.save("/kaggle/working/y_test.npy", y_test)

print("\n================================")
print("PREPROCESSING COMPLETE")
print("================================")

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("Classes:", len(classes))
print("Mean   :", mean.shape)
print("Std    :", std.shape)

Number of classes: 262
Encoded labels: (4284,)

Train:
X_train: (3427, 40, 225)
y_train: (3427,)

Test:
X_test: (857, 40, 225)
y_test: (857,)

PREPROCESSING COMPLETE
X_train: (3427, 40, 225)
X_test : (857, 40, 225)
Classes: 262
Mean   : (1, 1, 225)
Std    : (1, 1, 225)


In [64]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import numpy as np
import json
from sklearn.utils.class_weight import compute_class_weight

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

# --------------------------------------------------
# Load data
# --------------------------------------------------

X_train = np.load("/kaggle/working/X_train.npy")
X_test  = np.load("/kaggle/working/X_test.npy")
y_train = np.load("/kaggle/working/y_train.npy")
y_test  = np.load("/kaggle/working/y_test.npy")

with open("/kaggle/working/classes.json", "r") as f:
    classes_dict = json.load(f)

NUM_CLASSES = len(classes_dict)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("Classes:", NUM_CLASSES)

# --------------------------------------------------
# Class weights
# --------------------------------------------------

classes = np.unique(y_train)

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(
    zip(classes, weights)
)

print("Class weights created:", len(class_weights))

# --------------------------------------------------
# Build BiGRU
# --------------------------------------------------

model = keras.Sequential([
    
    layers.Input(shape=(40, 225)),

    layers.Bidirectional(
        layers.GRU(
            128,
            return_sequences=True
        )
    ),

    layers.Dropout(0.30),

    layers.Bidirectional(
        layers.GRU(
            64,
            return_sequences=False
        )
    ),

    layers.Dropout(0.30),

    layers.Dense(
        128,
        activation="relu"
    ),

    layers.Dropout(0.30),

    layers.Dense(
        NUM_CLASSES,
        activation="softmax"
    )
])

# --------------------------------------------------
# Compile
# --------------------------------------------------

model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=1e-3
    ),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

TensorFlow: 2.19.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
X_train: (3427, 40, 225)
X_test : (857, 40, 225)
Classes: 262
Class weights created: 262


I0000 00:00:1791486093.335329      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1791486093.341796      57 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 40, 256)        │       272,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 40, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 128)            │       123,648 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 262)            │        33,798 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 446,598 (1.70 MB)

 Trainable params: 446,598 (1.70 MB)

 Non-trainable params: 0 (0.00 B)

In [65]:
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    EarlyStopping,
    ReduceLROnPlateau
)

checkpoint = ModelCheckpoint(
    "/kaggle/working/best_isl_bigru.keras",
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    verbose=1
)

early_stopping = EarlyStopping(
    monitor="val_accuracy",
    mode="max",
    patience=8,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    min_lr=1e-6,
    verbose=1
)

history = model.fit(
    X_train,
    y_train,

    validation_split=0.15,

    epochs=50,
    batch_size=32,

    class_weight=class_weights,

    callbacks=[
        checkpoint,
        early_stopping,
        reduce_lr
    ],

    shuffle=True
)

Epoch 1/50


I0000 00:00:1791486161.158131   43324 cuda_dnn.cc:529] Loaded cuDNN version 91002


90/91 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - accuracy: 0.0070 - loss: 5.6214
Epoch 1: val_accuracy improved from -inf to 0.00583, saving model to /kaggle/working/best_isl_bigru.keras
91/91 ━━━━━━━━━━━━━━━━━━━━ 7s 22ms/step - accuracy: 0.0070 - loss: 5.6211 - val_accuracy: 0.0058 - val_loss: 5.4850 - learning_rate: 0.0010
Epoch 2/50
91/91 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.0148 - loss: 5.4127
Epoch 2: val_accuracy improved from 0.00583 to 0.01553, saving model to /kaggle/working/best_isl_bigru.keras
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.0148 - loss: 5.4116 - val_accuracy: 0.0155 - val_loss: 5.1387 - learning_rate: 0.0010
Epoch 3/50
90/91 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.0289 - loss: 4.9294
Epoch 3: val_accuracy improved from 0.01553 to 0.02913, saving model to /kaggle/working/best_isl_bigru.keras
91/91 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.0290 - loss: 4.9268 - val_accuracy: 0.0291 - val_loss: 4.5897 - learning_rate: 0.0010
Epoch 4/50
9

In [66]:
from tensorflow import keras
import numpy as np

# Load best checkpoint
best_model = keras.models.load_model(
    "/kaggle/working/best_isl_bigru.keras"
)

# Evaluate completely unseen test set
test_loss, test_accuracy = best_model.evaluate(
    X_test,
    y_test,
    batch_size=32,
    verbose=1
)

print("\n================================")
print("FINAL TEST RESULT")
print("================================")
print(f"Test Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_accuracy:.4%}")

27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - accuracy: 0.8240 - loss: 0.6072

FINAL TEST RESULT
Test Loss     : 0.7708
Test Accuracy : 81.3302%


In [67]:
from sklearn.metrics import (
    classification_report,
    f1_score,
    confusion_matrix
)
import numpy as np

# Predictions
y_prob = best_model.predict(
    X_test,
    batch_size=32,
    verbose=1
)

y_pred = np.argmax(y_prob, axis=1)

# --------------------------------------------------
# Accuracy
# --------------------------------------------------

accuracy = np.mean(y_pred == y_test)

# --------------------------------------------------
# F1 scores
# --------------------------------------------------

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro",
    zero_division=0
)

weighted_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted",
    zero_division=0
)

print("\n================================")
print("TEST METRICS")
print("================================")

print(f"Accuracy    : {accuracy:.4%}")
print(f"Macro F1    : {macro_f1:.4f}")
print(f"Weighted F1 : {weighted_f1:.4f}")

# --------------------------------------------------
# Classification report
# --------------------------------------------------

with open("/kaggle/working/classes.json", "r") as f:
    classes_dict = json.load(f)

target_names = [
    classes_dict[str(i)]
    for i in range(len(classes_dict))
]

report = classification_report(
    y_test,
    y_pred,
    target_names=target_names,
    zero_division=0
)

print("\nCLASSIFICATION REPORT")
print(report)

# Save report
with open(
    "/kaggle/working/classification_report.txt",
    "w"
) as f:
    f.write(report)

27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step

TEST METRICS
Accuracy    : 81.3302%
Macro F1    : 0.8064
Weighted F1 : 0.8045

CLASSIFICATION REPORT
                precision    recall  f1-score   support

         actor       0.75      1.00      0.86         3
         adult       1.00      1.00      1.00         4
     afternoon       0.67      0.67      0.67         3
         alive       1.00      1.00      1.00         2
       alright       0.67      1.00      0.80         4
        animal       1.00      1.00      1.00         4
        artist       1.00      0.33      0.50         3
        attack       1.00      1.00      1.00         3
        author       0.60      1.00      0.75         3
          baby       0.75      0.75      0.75         4
           bad       0.67      1.00      0.80         4
           bag       1.00      0.33      0.50         3
          ball       1.00      1.00      1.00         3
          bank       1.00      0.75      0.86         4
      bathroom   

In [68]:
from pathlib import Path
import shutil
import json

PACKAGE = Path("/kaggle/working/signora_model")
PACKAGE.mkdir(exist_ok=True)

# Model
shutil.copy(
    "/kaggle/working/best_isl_bigru.keras",
    PACKAGE / "best_isl_bigru.keras"
)

# Class mapping
shutil.copy(
    "/kaggle/working/classes.json",
    PACKAGE / "classes.json"
)

# Normalization
shutil.copy(
    "/kaggle/working/mean.npy",
    PACKAGE / "mean.npy"
)

shutil.copy(
    "/kaggle/working/std.npy",
    PACKAGE / "std.npy"
)

# MediaPipe model
shutil.copy(
    "/kaggle/working/holistic_landmarker.task",
    PACKAGE / "holistic_landmarker.task"
)

print("================================")
print("SIGNORA MODEL PACKAGE")
print("================================")

for f in PACKAGE.iterdir():
    print(f.name, f"{f.stat().st_size / (1024**2):.2f} MB")

SIGNORA MODEL PACKAGE
std.npy 0.00 MB
classes.json 0.00 MB
mean.npy 0.00 MB
best_isl_bigru.keras 5.17 MB
holistic_landmarker.task 13.05 MB


In [69]:
from pathlib import Path
import shutil
import json
import numpy as np
import tensorflow as tf
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score
)

# ============================================================
# SIGNORA EXPORT PACKAGE
# ============================================================

WORK = Path("/kaggle/working")
MODEL_DIR = WORK / "signora_model"
EXPORT_DIR = WORK / "SignOra_Final_Export"

EXPORT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Required inference files
# ------------------------------------------------------------

required_files = [
    "best_isl_bigru.keras",
    "holistic_landmarker.task",
    "classes.json",
    "mean.npy",
    "std.npy",
]

print("Checking required files...\n")

for filename in required_files:
    src = MODEL_DIR / filename

    if src.exists():
        shutil.copy2(src, EXPORT_DIR / filename)
        size_mb = src.stat().st_size / (1024 * 1024)
        print(f"✓ {filename:<30} {size_mb:.2f} MB")
    else:
        print(f"✗ MISSING: {filename}")


# ------------------------------------------------------------
# 2. Generate classification_report.txt
# ------------------------------------------------------------

print("\nGenerating classification report...")

X_test_path = WORK / "X_test.npy"
y_test_path = WORK / "y_test.npy"

if X_test_path.exists() and y_test_path.exists():

    X_test = np.load(X_test_path)
    y_test = np.load(y_test_path)

    # Load classes
    classes_path = EXPORT_DIR / "classes.json"

    with open(classes_path, "r", encoding="utf-8") as f:
        classes_dict = json.load(f)

    # Convert {"0": "adult", ...} to ordered list
    classes = [
        classes_dict[str(i)]
        for i in range(len(classes_dict))
    ]

    # Load trained model
    model = tf.keras.models.load_model(
        EXPORT_DIR / "best_isl_bigru.keras"
    )

    # Predict
    y_prob = model.predict(
        X_test,
        batch_size=32,
        verbose=1
    )

    y_pred = np.argmax(y_prob, axis=1)

    # Metrics
    accuracy = accuracy_score(y_test, y_pred)
    macro_f1 = f1_score(
        y_test,
        y_pred,
        average="macro",
        zero_division=0
    )
    weighted_f1 = f1_score(
        y_test,
        y_pred,
        average="weighted",
        zero_division=0
    )

    # Classification report
    report = classification_report(
        y_test,
        y_pred,
        labels=np.arange(len(classes)),
        target_names=classes,
        zero_division=0
    )

    report_text = f"""
============================================================
SIGNORA - INDIAN SIGN LANGUAGE RECOGNITION
MODEL EVALUATION REPORT
============================================================

Model:
BiGRU + MediaPipe Holistic Landmarks

Classes:
{len(classes)}

Test samples:
{len(y_test)}

Sequence length:
40 frames

Features per frame:
225

============================================================
OVERALL METRICS
============================================================

Test Accuracy : {accuracy * 100:.2f}%
Macro F1      : {macro_f1:.4f}
Weighted F1   : {weighted_f1:.4f}

============================================================
CLASSIFICATION REPORT
============================================================

{report}

============================================================
END OF REPORT
============================================================
"""

    report_path = EXPORT_DIR / "classification_report.txt"

    with open(report_path, "w", encoding="utf-8") as f:
        f.write(report_text)

    print(f"\n✓ classification_report.txt")
    print(f"  Accuracy   : {accuracy * 100:.2f}%")
    print(f"  Macro F1   : {macro_f1:.4f}")
    print(f"  Weighted F1: {weighted_f1:.4f}")

else:
    print("⚠ X_test.npy or y_test.npy not found.")
    print("Skipping automatic report generation.")


# ------------------------------------------------------------
# 3. Create requirements.txt
# ------------------------------------------------------------

requirements = """tensorflow
numpy
scikit-learn
mediapipe
opencv-python
fastapi
uvicorn
python-multipart
"""

with open(EXPORT_DIR / "requirements.txt", "w") as f:
    f.write(requirements)

print("✓ requirements.txt")


# ------------------------------------------------------------
# 4. Create README
# ------------------------------------------------------------

readme = """# SignOra - Indian Sign Language Recognition

## Model

BiGRU classifier using MediaPipe Holistic landmarks.

## Input

40 frames × 225 features per frame.

### Features per frame

- Pose: 33 × 3 = 99
- Left hand: 21 × 3 = 63
- Right hand: 21 × 3 = 63

Total = 225 features.

## Required inference files

- best_isl_bigru.keras
- holistic_landmarker.task
- classes.json
- mean.npy
- std.npy

## Prediction pipeline

Video/Webcam
    ↓
MediaPipe Holistic
    ↓
Pose + Left Hand + Right Hand
    ↓
225 features/frame
    ↓
40-frame sequence
    ↓
Normalization
    ↓
BiGRU
    ↓
262-class prediction
    ↓
Indian Sign Language word

## Evaluation

See classification_report.txt.

"""

with open(EXPORT_DIR / "README.md", "w", encoding="utf-8") as f:
    f.write(readme)

print("✓ README.md")


# ------------------------------------------------------------
# 5. Copy notebook source if available
# ------------------------------------------------------------

notebook_source = (
    WORK /
    ".virtual_documents" /
    "__notebook_source__.ipynb"
)

if notebook_source.exists():
    shutil.copy2(
        notebook_source,
        EXPORT_DIR / "SignOra_MediaPipe_BiGRU.ipynb"
    )
    print("✓ SignOra_MediaPipe_BiGRU.ipynb")


# ------------------------------------------------------------
# 6. List final export
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("FINAL EXPORT")
print("=" * 60)

total_size = 0

for file in sorted(EXPORT_DIR.iterdir()):
    if file.is_file():
        size_mb = file.stat().st_size / (1024 * 1024)
        total_size += size_mb
        print(f"{file.name:<40} {size_mb:>8.2f} MB")

print("-" * 60)
print(f"Total: {total_size:.2f} MB")


# ------------------------------------------------------------
# 7. Create ZIP
# ------------------------------------------------------------

zip_base = WORK / "SignOra_Final_Export"

zip_path = shutil.make_archive(
    str(zip_base),
    "zip",
    root_dir=EXPORT_DIR
)

zip_size = Path(zip_path).stat().st_size / (1024 * 1024)

print("\n" + "=" * 60)
print("ZIP CREATED")
print("=" * 60)
print(zip_path)
print(f"ZIP size: {zip_size:.2f} MB")

Checking required files...

✓ best_isl_bigru.keras           5.17 MB
✓ holistic_landmarker.task       13.05 MB
✓ classes.json                   0.00 MB
✓ mean.npy                       0.00 MB
✓ std.npy                        0.00 MB

Generating classification report...
27/27 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step

✓ classification_report.txt
  Accuracy   : 81.33%
  Macro F1   : 0.8064
  Weighted F1: 0.8045
✓ requirements.txt
✓ README.md
✓ SignOra_MediaPipe_BiGRU.ipynb

FINAL EXPORT
README.md                                    0.00 MB
SignOra_MediaPipe_BiGRU.ipynb                0.02 MB
best_isl_bigru.keras                         5.17 MB
classes.json                                 0.00 MB
classification_report.txt                    0.01 MB
holistic_landmarker.task                    13.05 MB
mean.npy                                     0.00 MB
requirements.txt                             0.00 MB
std.npy                                      0.00 MB
----------------------------------------

In [72]:
from pathlib import Path
import json
import cv2
import numpy as np
import tensorflow as tf
import mediapipe as mp

# ============================================================
# SIGNORA - TEST ONE VIDEO
# ============================================================

MODEL_DIR = Path("/kaggle/working/signora_model")

MODEL_PATH = MODEL_DIR / "best_isl_bigru.keras"
TASK_PATH = MODEL_DIR / "holistic_landmarker.task"
CLASSES_PATH = MODEL_DIR / "classes.json"
MEAN_PATH = MODEL_DIR / "mean.npy"
STD_PATH = MODEL_DIR / "std.npy"

DATASET_PATH = Path("/kaggle/input/datasets/swaptr/isl-video")

# ------------------------------------------------------------
# Find one video
# ------------------------------------------------------------

videos = []

for ext in ["*.MOV", "*.mov", "*.MP4", "*.mp4"]:
    videos.extend(DATASET_PATH.rglob(ext))

TEST_VIDEO = videos[0]

print("Testing:")
print(TEST_VIDEO)

# ------------------------------------------------------------
# Load model
# ------------------------------------------------------------

model = tf.keras.models.load_model(MODEL_PATH)

with open(CLASSES_PATH, "r", encoding="utf-8") as f:
    class_dict = json.load(f)

classes = [class_dict[str(i)] for i in range(len(class_dict))]

mean = np.load(MEAN_PATH)
std = np.load(STD_PATH)

print("Classes:", len(classes))
print("Model loaded successfully")

# ------------------------------------------------------------
# MediaPipe
# ------------------------------------------------------------

options = mp.tasks.vision.HolisticLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(
        model_asset_path=str(TASK_PATH)
    ),
    running_mode=mp.tasks.vision.RunningMode.VIDEO,
    min_face_detection_confidence=0.5,
    min_face_suppression_threshold=0.5,
    min_face_landmarks_confidence=0.5,
    min_pose_detection_confidence=0.5,
    min_pose_suppression_threshold=0.5,
    min_pose_landmarks_confidence=0.5,
    min_hand_landmarks_confidence=0.5
)

# ------------------------------------------------------------
# Extract 225 features per frame
# ------------------------------------------------------------

features = []

cap = cv2.VideoCapture(str(TEST_VIDEO))

frame_number = 0

with mp.tasks.vision.HolisticLandmarker.create_from_options(options) as landmarker:

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        h, w = frame.shape[:2]

        # Same width as training
        if w > 480:
            new_h = int(h * 480 / w)
            frame = cv2.resize(frame, (480, new_h))
            h, w = frame.shape[:2]

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        timestamp_ms = frame_number * 33

        result = landmarker.detect_for_video(
            image,
            timestamp_ms
        )

        frame_values = []

        # ====================================================
        # POSE - 33 landmarks
        # ====================================================

        pose = result.pose_landmarks

        if pose is not None and len(pose) > 0:

            # MediaPipe may return one pose landmark list
            if hasattr(pose[0], "x"):
                pose_list = pose
            else:
                pose_list = pose[0]

            for i in range(33):

                if i < len(pose_list):

                    lm = pose_list[i]

                    frame_values.append(lm.x)
                    frame_values.append(lm.y * (h / w))
                    frame_values.append(lm.z)

                else:

                    frame_values.extend(
                        [np.nan, np.nan, np.nan]
                    )

        else:

            frame_values.extend(
                [np.nan] * 99
            )

        # ====================================================
        # LEFT HAND - 21 landmarks
        # ====================================================

        left = result.left_hand_landmarks

        if left is not None and len(left) > 0:

            if hasattr(left[0], "x"):
                left_list = left
            else:
                left_list = left[0]

            for i in range(21):

                if i < len(left_list):

                    lm = left_list[i]

                    frame_values.append(lm.x)
                    frame_values.append(lm.y * (h / w))
                    frame_values.append(lm.z)

                else:

                    frame_values.extend(
                        [np.nan, np.nan, np.nan]
                    )

        else:

            frame_values.extend(
                [np.nan] * 63
            )

        # ====================================================
        # RIGHT HAND - 21 landmarks
        # ====================================================

        right = result.right_hand_landmarks

        if right is not None and len(right) > 0:

            if hasattr(right[0], "x"):
                right_list = right
            else:
                right_list = right[0]

            for i in range(21):

                if i < len(right_list):

                    lm = right_list[i]

                    frame_values.append(lm.x)
                    frame_values.append(lm.y * (h / w))
                    frame_values.append(lm.z)

                else:

                    frame_values.extend(
                        [np.nan, np.nan, np.nan]
                    )

        else:

            frame_values.extend(
                [np.nan] * 63
            )

        # ----------------------------------------------------
        # Check
        # ----------------------------------------------------

        if len(frame_values) != 225:

            raise RuntimeError(
                f"Expected 225 features, got {len(frame_values)}"
            )

        features.append(frame_values)

        frame_number += 1

cap.release()

# ------------------------------------------------------------
# Convert to numpy
# ------------------------------------------------------------

X = np.asarray(features, dtype=np.float32)

print()
print("Extracted shape:", X.shape)

if len(X) == 0:
    raise RuntimeError("No frames extracted!")

# ------------------------------------------------------------
# Convert to 40 frames
# ------------------------------------------------------------

if len(X) >= 40:

    indices = np.linspace(
        0,
        len(X) - 1,
        40
    ).astype(int)

    X = X[indices]

else:

    padding = np.repeat(
        X[-1][None, :],
        40 - len(X),
        axis=0
    )

    X = np.concatenate(
        [X, padding],
        axis=0
    )

print("40-frame shape:", X.shape)

# ------------------------------------------------------------
# Replace missing landmarks
# ------------------------------------------------------------

X = np.nan_to_num(
    X,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

# ------------------------------------------------------------
# Normalize
# ------------------------------------------------------------

X = (X - mean) / std

# Add batch dimension
X = X.astype(np.float32)
X = np.expand_dims(X, axis=0)

print("Model input:", X.shape)

# ------------------------------------------------------------
# Predict
# ------------------------------------------------------------

probabilities = model.predict(
    X,
    verbose=0
)[0]

top5 = np.argsort(probabilities)[::-1][:5]

print()
print("=" * 55)
print("SIGNORA PREDICTION")
print("=" * 55)

for rank, index in enumerate(top5, 1):

    print(
        f"{rank}. {classes[index]:25s} "
        f"{probabilities[index] * 100:.2f}%"
    )

print("=" * 55)

best_index = top5[0]

print()
print("FINAL PREDICTION :", classes[best_index])
print(
    "CONFIDENCE       : "
    f"{probabilities[best_index] * 100:.2f}%"
)

Testing:
/kaggle/input/datasets/swaptr/isl-video/adjectives/female/MVI_9767.MOV
Classes: 262
Model loaded successfully


W0000 00:00:1791487603.890446   44688 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791487603.919911   44689 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791487603.923815   44687 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791487603.924923   44688 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791487603.926176   44689 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791487603.950360   44690 inference_feedback_manager.cc:121] Feedback manager 


Extracted shape: (47, 225)
40-frame shape: (40, 225)
Model input: (1, 1, 40, 225)


ValueError: Exception encountered when calling Sequential.call().

[1mInvalid input shape for input Tensor("data:0", shape=(1, 1, 40, 225), dtype=float32). Expected shape (None, 40, 225), but input has incompatible shape (1, 1, 40, 225)[0m

Arguments received by Sequential.call():
  • inputs=tf.Tensor(shape=(1, 1, 40, 225), dtype=float32)
  • training=False
  • mask=None
  • kwargs=<class 'inspect._empty'>

In [73]:
# ============================================================
# FIX NORMALIZATION SHAPE + PREDICTION
# ============================================================

print("Before normalization:", X.shape)

# ------------------------------------------------------------
# Inspect saved normalization arrays
# ------------------------------------------------------------

print("mean.npy shape:", mean.shape)
print("std.npy shape :", std.shape)

# Flatten normalization statistics to 225 features
mean_fixed = np.asarray(mean).reshape(-1)
std_fixed = np.asarray(std).reshape(-1)

print("Flattened mean:", mean_fixed.shape)
print("Flattened std :", std_fixed.shape)

# Make sure they contain exactly 225 features
if mean_fixed.size != 225:
    raise ValueError(
        f"mean.npy contains {mean_fixed.size} values, "
        f"but expected 225"
    )

if std_fixed.size != 225:
    raise ValueError(
        f"std.npy contains {std_fixed.size} values, "
        f"but expected 225"
    )

# Reshape to broadcast correctly over:
# (40 frames, 225 features)
mean_fixed = mean_fixed.reshape(1, 225)
std_fixed = std_fixed.reshape(1, 225)

# ------------------------------------------------------------
# Normalize
# ------------------------------------------------------------

X = np.nan_to_num(
    X,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

X = (X - mean_fixed) / std_fixed

X = X.astype(np.float32)

print("After normalization:", X.shape)

# ------------------------------------------------------------
# Add ONLY ONE batch dimension
# ------------------------------------------------------------

if X.ndim == 2:
    X = np.expand_dims(X, axis=0)

print("Final model input:", X.shape)

# Expected:
# (1, 40, 225)

if X.shape != (1, 40, 225):
    raise ValueError(
        f"Wrong model input shape: {X.shape}. "
        f"Expected (1, 40, 225)"
    )

# ------------------------------------------------------------
# Prediction
# ------------------------------------------------------------

probabilities = model.predict(
    X,
    verbose=0
)[0]

top5 = np.argsort(
    probabilities
)[::-1][:5]

print()
print("=" * 60)
print("                 SIGNORA PREDICTION")
print("=" * 60)

for rank, index in enumerate(top5, 1):

    print(
        f"{rank}. "
        f"{classes[index]:25s} "
        f"{probabilities[index] * 100:6.2f}%"
    )

print("=" * 60)

best_index = top5[0]

print()
print("FINAL PREDICTION :", classes[best_index])
print(
    "CONFIDENCE       : "
    f"{probabilities[best_index] * 100:.2f}%"
)

Before normalization: (1, 1, 40, 225)
mean.npy shape: (1, 1, 225)
std.npy shape : (1, 1, 225)
Flattened mean: (225,)
Flattened std : (225,)
After normalization: (1, 1, 40, 225)
Final model input: (1, 1, 40, 225)


ValueError: Wrong model input shape: (1, 1, 40, 225). Expected (1, 40, 225)

In [74]:
# ============================================================
# FINAL PREDICTION - FIX CURRENT X
# ============================================================

print("Current X:", X.shape)

# Remove the two unnecessary dimensions
X_test_video = np.squeeze(X)

print("After squeeze:", X_test_video.shape)

# We need exactly:
# (40, 225)

if X_test_video.shape != (40, 225):
    raise ValueError(
        f"Expected (40, 225), got {X_test_video.shape}"
    )

# Add exactly ONE batch dimension
X_test_video = X_test_video[np.newaxis, :, :]

print("Final model input:", X_test_video.shape)

# Expected:
# (1, 40, 225)

# ============================================================
# PREDICTION
# ============================================================

probabilities = model.predict(
    X_test_video,
    verbose=0
)[0]

top5 = np.argsort(probabilities)[::-1][:5]

print()
print("=" * 60)
print("                 SIGNORA PREDICTION")
print("=" * 60)

for rank, index in enumerate(top5, 1):
    print(
        f"{rank}. "
        f"{classes[index]:25s} "
        f"{probabilities[index] * 100:.2f}%"
    )

print("=" * 60)

best_index = top5[0]

print()
print("FINAL PREDICTION :", classes[best_index])
print(
    "CONFIDENCE       : "
    f"{probabilities[best_index] * 100:.2f}%"
)

Current X: (1, 1, 40, 225)
After squeeze: (40, 225)
Final model input: (1, 40, 225)

                 SIGNORA PREDICTION
1. patient                   53.52%
2. screen                    29.97%
3. female                    10.19%
4. cellphone                 1.72%
5. hospital                  1.14%

FINAL PREDICTION : patient
CONFIDENCE       : 53.52%


In [75]:
from pathlib import Path

true_label = Path(TEST_VIDEO).parent.name

print("True label :", true_label)
print("Prediction  :", classes[best_index])

if true_label == classes[best_index]:
    print("✅ CORRECT PREDICTION")
else:
    print("❌ INCORRECT PREDICTION")

True label : female
Prediction  : patient
❌ INCORRECT PREDICTION


In [76]:
# Find a video whose true label is "cat"
target_label = "cat"

matching_videos = [
    v for v in videos
    if Path(v).parent.name == target_label
]

print("Number of videos:", len(matching_videos))

TEST_VIDEO = matching_videos[0]

print("Testing:", TEST_VIDEO)
print("True label:", Path(TEST_VIDEO).parent.name)

Number of videos: 20
Testing: /kaggle/input/datasets/swaptr/isl-video/animals/cat/MVI_2982.MOV
True label: cat


In [80]:
# ============================================================
# SIGNORA - COMPLETE SINGLE VIDEO INFERENCE TEST
# ============================================================
#
# This cell:
# 1. Selects ONE video
# 2. Extracts MediaPipe Holistic landmarks
# 3. Uses Pose + Left Hand + Right Hand = 225 features
# 4. Converts video to 40 frames
# 5. Normalizes using mean.npy / std.npy
# 6. Runs best_isl_bigru.keras
# 7. Shows Top-5 predictions
# 8. Compares prediction with the actual folder label
#
# IMPORTANT:
# This does NOT retrain anything.
# This does NOT process the full dataset.
# ============================================================

from pathlib import Path
import json
import cv2
import numpy as np
import tensorflow as tf
import mediapipe as mp

# ============================================================
# 1. PATHS
# ============================================================

MODEL_DIR = Path("/kaggle/working/signora_model")

MODEL_PATH = MODEL_DIR / "best_isl_bigru.keras"
TASK_PATH = MODEL_DIR / "holistic_landmarker.task"
CLASSES_PATH = MODEL_DIR / "classes.json"
MEAN_PATH = MODEL_DIR / "mean.npy"
STD_PATH = MODEL_DIR / "std.npy"

DATASET_PATH = Path(
    "/kaggle/input/datasets/swaptr/isl-video"
)

# ============================================================
# 2. CHECK REQUIRED FILES
# ============================================================

print("=" * 65)
print("CHECKING SIGNORA FILES")
print("=" * 65)

required_files = [
    MODEL_PATH,
    TASK_PATH,
    CLASSES_PATH,
    MEAN_PATH,
    STD_PATH
]

for file_path in required_files:

    if not file_path.exists():
        raise FileNotFoundError(
            f"Required file not found:\n{file_path}"
        )

    size_mb = file_path.stat().st_size / (1024 * 1024)

    print(
        f"✓ {file_path.name:<30} "
        f"{size_mb:.2f} MB"
    )

# ============================================================
# 3. SELECT ONE VIDEO
# ============================================================
#
# We deliberately select a known "cat" video.
# This avoids accidentally testing the same old video.
# ============================================================

TARGET_LABEL = "cat"

print("\n" + "=" * 65)
print("SEARCHING FOR TEST VIDEO")
print("=" * 65)

matching_videos = []

for ext in [
    "*.MOV",
    "*.mov",
    "*.MP4",
    "*.mp4"
]:
    matching_videos.extend(
        DATASET_PATH.rglob(ext)
    )

matching_videos = [
    v
    for v in matching_videos
    if v.parent.name == TARGET_LABEL
]

if len(matching_videos) == 0:
    raise RuntimeError(
        f"No videos found for class: {TARGET_LABEL}"
    )

# Use the first cat video
TEST_VIDEO = matching_videos[0]

TRUE_LABEL = TEST_VIDEO.parent.name

print("Target class      :", TARGET_LABEL)
print("Number of videos  :", len(matching_videos))
print("Selected video    :", TEST_VIDEO)
print("True label        :", TRUE_LABEL)

# ============================================================
# 4. LOAD MODEL
# ============================================================

print("\n" + "=" * 65)
print("LOADING MODEL")
print("=" * 65)

model = tf.keras.models.load_model(
    MODEL_PATH
)

print("✓ BiGRU model loaded")
print("Model input shape :", model.input_shape)
print("Model output shape:", model.output_shape)

# ============================================================
# 5. LOAD CLASSES
# ============================================================

with open(
    CLASSES_PATH,
    "r",
    encoding="utf-8"
) as f:

    class_dict = json.load(f)

classes = [
    class_dict[str(i)]
    for i in range(len(class_dict))
]

print("Number of classes :", len(classes))

# ============================================================
# 6. LOAD NORMALIZATION PARAMETERS
# ============================================================

mean = np.load(MEAN_PATH)
std = np.load(STD_PATH)

print("Original mean shape:", mean.shape)
print("Original std shape :", std.shape)

# Convert to exactly 225 feature values
mean = mean.reshape(225)
std = std.reshape(225)

print("Final mean shape   :", mean.shape)
print("Final std shape    :", std.shape)

# Prevent division by zero
std = np.where(
    std < 1e-6,
    1e-6,
    std
)

# ============================================================
# 7. MEDIA PIPE HOLISTIC
# ============================================================

print("\n" + "=" * 65)
print("STARTING MEDIAPIPE EXTRACTION")
print("=" * 65)

options = mp.tasks.vision.HolisticLandmarkerOptions(

    base_options=mp.tasks.BaseOptions(
        model_asset_path=str(TASK_PATH)
    ),

    running_mode=(
        mp.tasks.vision.RunningMode.VIDEO
    ),

    min_face_detection_confidence=0.5,
    min_face_suppression_threshold=0.5,
    min_face_landmarks_confidence=0.5,

    min_pose_detection_confidence=0.5,
    min_pose_suppression_threshold=0.5,
    min_pose_landmarks_confidence=0.5,

    min_hand_landmarks_confidence=0.5
)

# ============================================================
# 8. OPEN VIDEO
# ============================================================

cap = cv2.VideoCapture(
    str(TEST_VIDEO)
)

if not cap.isOpened():

    raise RuntimeError(
        f"Could not open video:\n{TEST_VIDEO}"
    )

total_frames = int(
    cap.get(cv2.CAP_PROP_FRAME_COUNT)
)

fps = cap.get(
    cv2.CAP_PROP_FPS
)

print("Video frames:", total_frames)
print("Video FPS   :", fps)

# ============================================================
# 9. EXTRACT FEATURES
# ============================================================

all_frames = []

frame_number = 0

with mp.tasks.vision.HolisticLandmarker.create_from_options(
    options
) as landmarker:

    while True:

        ret, frame = cap.read()

        if not ret:
            break

        # ----------------------------------------------------
        # Original dimensions
        # ----------------------------------------------------

        h, w = frame.shape[:2]

        # ----------------------------------------------------
        # Same resizing used during training
        # ----------------------------------------------------

        if w > 480:

            new_h = int(
                h * 480 / w
            )

            frame = cv2.resize(
                frame,
                (480, new_h)
            )

            h, w = frame.shape[:2]

        # ----------------------------------------------------
        # BGR → RGB
        # ----------------------------------------------------

        rgb = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )

        # ----------------------------------------------------
        # MediaPipe image
        # ----------------------------------------------------

        image = mp.Image(
            image_format=mp.ImageFormat.SRGB,
            data=rgb
        )

        # ----------------------------------------------------
        # Monotonically increasing timestamp
        # ----------------------------------------------------

        timestamp_ms = (
            frame_number * 33
        )

        result = landmarker.detect_for_video(
            image,
            timestamp_ms
        )

        frame_values = []

        # ====================================================
        # POSE
        # 33 landmarks × 3 = 99
        # ====================================================

        pose = result.pose_landmarks

        if pose is not None and len(pose) > 0:

            # MediaPipe can expose the pose as:
            # [landmark, landmark, ...]
            #
            # or:
            # [[landmark, landmark, ...]]
            #
            # Handle both.

            if hasattr(
                pose[0],
                "x"
            ):

                pose_list = pose

            else:

                pose_list = pose[0]

            for i in range(33):

                if i < len(pose_list):

                    lm = pose_list[i]

                    frame_values.extend([
                        lm.x,
                        lm.y * (h / w),
                        lm.z
                    ])

                else:

                    frame_values.extend([
                        np.nan,
                        np.nan,
                        np.nan
                    ])

        else:

            frame_values.extend(
                [np.nan] * 99
            )

        # ====================================================
        # LEFT HAND
        # 21 landmarks × 3 = 63
        # ====================================================

        left_hand = (
            result.left_hand_landmarks
        )

        if (
            left_hand is not None
            and len(left_hand) > 0
        ):

            if hasattr(
                left_hand[0],
                "x"
            ):

                left_list = left_hand

            else:

                left_list = left_hand[0]

            for i in range(21):

                if i < len(left_list):

                    lm = left_list[i]

                    frame_values.extend([
                        lm.x,
                        lm.y * (h / w),
                        lm.z
                    ])

                else:

                    frame_values.extend([
                        np.nan,
                        np.nan,
                        np.nan
                    ])

        else:

            frame_values.extend(
                [np.nan] * 63
            )

        # ====================================================
        # RIGHT HAND
        # 21 landmarks × 3 = 63
        # ====================================================

        right_hand = (
            result.right_hand_landmarks
        )

        if (
            right_hand is not None
            and len(right_hand) > 0
        ):

            if hasattr(
                right_hand[0],
                "x"
            ):

                right_list = right_hand

            else:

                right_list = right_hand[0]

            for i in range(21):

                if i < len(right_list):

                    lm = right_list[i]

                    frame_values.extend([
                        lm.x,
                        lm.y * (h / w),
                        lm.z
                    ])

                else:

                    frame_values.extend([
                        np.nan,
                        np.nan,
                        np.nan
                    ])

        else:

            frame_values.extend(
                [np.nan] * 63
            )

        # ====================================================
        # VERIFY 225 FEATURES
        # ====================================================

        if len(frame_values) != 225:

            raise RuntimeError(
                f"Frame {frame_number}: "
                f"Expected 225 features, "
                f"got {len(frame_values)}"
            )

        all_frames.append(
            frame_values
        )

        frame_number += 1

cap.release()

# ============================================================
# 10. CONVERT TO NUMPY
# ============================================================

X = np.asarray(
    all_frames,
    dtype=np.float32
)

print("\nExtraction completed.")
print("Raw feature shape:", X.shape)

if X.shape[0] == 0:

    raise RuntimeError(
        "No frames were extracted."
    )

if X.shape[1] != 225:

    raise RuntimeError(
        f"Expected 225 features, "
        f"got {X.shape[1]}"
    )

# ============================================================
# 11. CONVERT TO EXACTLY 40 FRAMES
# ============================================================

if X.shape[0] >= 40:

    # Uniform temporal sampling
    indices = np.linspace(
        0,
        X.shape[0] - 1,
        40
    ).astype(int)

    X = X[indices]

else:

    # Pad using final frame
    padding_count = (
        40 - X.shape[0]
    )

    padding = np.repeat(
        X[-1][None, :],
        padding_count,
        axis=0
    )

    X = np.concatenate(
        [X, padding],
        axis=0
    )

print("After 40-frame conversion:", X.shape)

if X.shape != (40, 225):

    raise RuntimeError(
        f"Expected (40, 225), "
        f"got {X.shape}"
    )

# ============================================================
# 12. HANDLE MISSING LANDMARKS
# ============================================================

X = np.nan_to_num(
    X,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

# ============================================================
# 13. NORMALIZE
# ============================================================

X = (
    X - mean
) / std

X = X.astype(
    np.float32
)

print("After normalization:", X.shape)

# ============================================================
# 14. ADD ONE BATCH DIMENSION
# ============================================================

X_input = X[
    np.newaxis,
    :,
    :
]

print("FINAL MODEL INPUT:", X_input.shape)

if X_input.shape != (
    1,
    40,
    225
):

    raise RuntimeError(
        f"Model input must be "
        f"(1, 40, 225), "
        f"got {X_input.shape}"
    )

# ============================================================
# 15. PREDICTION
# ============================================================

print("\n" + "=" * 65)
print("RUNNING SIGNORA MODEL")
print("=" * 65)

probabilities = model.predict(
    X_input,
    verbose=0
)[0]

# ============================================================
# 16. TOP 5 PREDICTIONS
# ============================================================

top5_indices = np.argsort(
    probabilities
)[::-1][:5]

print("\n" + "=" * 65)
print("SIGNORA TOP-5 PREDICTIONS")
print("=" * 65)

for rank, index in enumerate(
    top5_indices,
    start=1
):

    print(
        f"{rank}. "
        f"{classes[index]:25s} "
        f"{probabilities[index] * 100:7.2f}%"
    )

# ============================================================
# 17. FINAL RESULT
# ============================================================

best_index = top5_indices[0]

predicted_label = classes[
    best_index
]

confidence = (
    probabilities[best_index]
    * 100
)

print("\n" + "=" * 65)
print("FINAL RESULT")
print("=" * 65)

print(
    "True label       :",
    TRUE_LABEL
)

print(
    "Predicted label  :",
    predicted_label
)

print(
    f"Confidence       : {confidence:.2f}%"
)

# ============================================================
# 18. CHECK CORRECTNESS
# ============================================================

print()

if (
    TRUE_LABEL.lower()
    == predicted_label.lower()
):

    print(
        "RESULT           : ✅ CORRECT"
    )

else:

    print(
        "RESULT           : ❌ INCORRECT"
    )

# ============================================================
# 19. CHECK WHETHER TRUE CLASS IS IN TOP 5
# ============================================================

true_index = None

for i, label in enumerate(classes):

    if label.lower() == TRUE_LABEL.lower():

        true_index = i
        break

if true_index is not None:

    true_probability = (
        probabilities[true_index]
        * 100
    )

    true_rank = (
        np.where(
            np.argsort(probabilities)[::-1]
            == true_index
        )[0][0]
        + 1
    )

    print()
    print(
        f"True class probability : "
        f"{true_probability:.2f}%"
    )

    print(
        f"True class rank       : "
        f"{true_rank}"
    )

# ============================================================
# 20. FINAL SUMMARY
# ============================================================

print("\n" + "=" * 65)
print("SIGNORA TEST SUMMARY")
print("=" * 65)

print(
    "Video              :",
    TEST_VIDEO.name
)

print(
    "True label         :",
    TRUE_LABEL
)

print(
    "Prediction         :",
    predicted_label
)

print(
    f"Confidence         : {confidence:.2f}%"
)

print(
    "Input shape        :",
    X_input.shape
)

print(
    "Features/frame     :",
    225
)

print(
    "Frames used        :",
    40
)

print(
    "Number of classes  :",
    len(classes)
)

print("=" * 65)

CHECKING SIGNORA FILES
✓ best_isl_bigru.keras           5.17 MB
✓ holistic_landmarker.task       13.05 MB
✓ classes.json                   0.00 MB
✓ mean.npy                       0.00 MB
✓ std.npy                        0.00 MB

SEARCHING FOR TEST VIDEO
Target class      : cat
Number of videos  : 20
Selected video    : /kaggle/input/datasets/swaptr/isl-video/animals/cat/MVI_2982.MOV
True label        : cat

LOADING MODEL
✓ BiGRU model loaded
Model input shape : (None, 40, 225)
Model output shape: (None, 262)
Number of classes : 262
Original mean shape: (1, 1, 225)
Original std shape : (1, 1, 225)
Final mean shape   : (225,)
Final std shape    : (225,)

STARTING MEDIAPIPE EXTRACTION
Video frames: 55
Video FPS   : 25.0


W0000 00:00:1791488027.117718   44737 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791488027.150669   44737 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791488027.154737   44739 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791488027.155705   44736 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791488027.157446   44737 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1791488027.171183   44739 inference_feedback_manager.cc:121] Feedback manager 


Extraction completed.
Raw feature shape: (55, 225)
After 40-frame conversion: (40, 225)
After normalization: (40, 225)
FINAL MODEL INPUT: (1, 40, 225)

RUNNING SIGNORA MODEL

SIGNORA TOP-5 PREDICTIONS
1. truck                       44.56%
2. cat                         42.65%
3. transportation               7.81%
4. cow                          3.54%
5. adult                        0.43%

FINAL RESULT
True label       : cat
Predicted label  : truck
Confidence       : 44.56%

RESULT           : ❌ INCORRECT

True class probability : 42.65%
True class rank       : 2

SIGNORA TEST SUMMARY
Video              : MVI_2982.MOV
True label         : cat
Prediction         : truck
Confidence         : 44.56%
Input shape        : (1, 40, 225)
Features/frame     : 225
Frames used        : 40
Number of classes  : 262


In [81]:
# ============================================================
# SIGNORA - EXACT TRAINING-PREPROCESSING INFERENCE TEST
# ============================================================
#
# IMPORTANT:
# This uses the parquet generated during the ORIGINAL
# 3-hour MediaPipe extraction.
#
# Therefore, we are testing the trained model using the
# SAME landmark representation that was used for training.
#
# No MediaPipe extraction is performed here.
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd
import tensorflow as tf

# ============================================================
# 1. PATHS
# ============================================================

MODEL_DIR = Path("/kaggle/working/signora_model")

MODEL_PATH = MODEL_DIR / "best_isl_bigru.keras"
CLASSES_PATH = MODEL_DIR / "classes.json"
MEAN_PATH = MODEL_DIR / "mean.npy"
STD_PATH = MODEL_DIR / "std.npy"

# Exact parquet generated during the original extraction
PARQUET_PATH = Path(
    "/kaggle/working/keypoints/"
    "animals/cat/MVI_2982.parquet"
)

# ============================================================
# 2. CHECK FILES
# ============================================================

print("=" * 65)
print("CHECKING FILES")
print("=" * 65)

for path in [
    MODEL_PATH,
    CLASSES_PATH,
    MEAN_PATH,
    STD_PATH,
    PARQUET_PATH
]:

    print(
        f"{'✓' if path.exists() else '✗'} "
        f"{path}"
    )

    if not path.exists():
        raise FileNotFoundError(
            f"Missing file:\n{path}"
        )

# ============================================================
# 3. LOAD MODEL
# ============================================================

model = tf.keras.models.load_model(
    MODEL_PATH
)

print("\nModel input :", model.input_shape)
print("Model output:", model.output_shape)

# ============================================================
# 4. LOAD CLASS NAMES
# ============================================================

with open(
    CLASSES_PATH,
    "r",
    encoding="utf-8"
) as f:

    class_dict = json.load(f)

classes = [
    class_dict[str(i)]
    for i in range(len(class_dict))
]

print("Classes:", len(classes))

# ============================================================
# 5. LOAD NORMALIZATION
# ============================================================

mean = np.load(MEAN_PATH)
std = np.load(STD_PATH)

print("\nOriginal mean shape:", mean.shape)
print("Original std shape :", std.shape)

mean = mean.reshape(225)
std = std.reshape(225)

std = np.maximum(
    std,
    1e-6
)

print("Mean shape:", mean.shape)
print("Std shape :", std.shape)

# ============================================================
# 6. LOAD ORIGINAL PARQUET
# ============================================================

print("\n" + "=" * 65)
print("LOADING ORIGINAL TRAINING LANDMARKS")
print("=" * 65)

df = pd.read_parquet(
    PARQUET_PATH
)

print("Parquet shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nLandmark types:")
print(df["type"].value_counts())

print(
    "\nNumber of frames:",
    df["frame"].nunique()
)

# ============================================================
# 7. BUILD EXACT 225-FEATURE REPRESENTATION
# ============================================================
#
# Training used:
#
# Pose       = 33 × 3 = 99
# Left hand  = 21 × 3 = 63
# Right hand = 21 × 3 = 63
#
# Total = 225
#
# Face is deliberately NOT used.
# ============================================================

POSE_COUNT = 33
HAND_COUNT = 21

FEATURE_TYPES = [
    ("pose", POSE_COUNT),
    ("left_hand", HAND_COUNT),
    ("right_hand", HAND_COUNT)
]

frames = sorted(
    df["frame"].unique()
)

print("\nFrames found:", len(frames))

frame_features = []

for frame_id in frames:

    frame_df = df[
        df["frame"] == frame_id
    ]

    values = []

    # --------------------------------------------------------
    # Pose
    # --------------------------------------------------------

    pose_df = frame_df[
        frame_df["type"] == "pose"
    ].sort_values(
        "landmark_index"
    )

    for i in range(POSE_COUNT):

        row = pose_df[
            pose_df["landmark_index"] == i
        ]

        if len(row) > 0:

            r = row.iloc[0]

            values.extend([
                r["x"],
                r["y"],
                r["z"]
            ])

        else:

            values.extend([
                np.nan,
                np.nan,
                np.nan
            ])

    # --------------------------------------------------------
    # Left hand
    # --------------------------------------------------------

    left_df = frame_df[
        frame_df["type"] == "left_hand"
    ].sort_values(
        "landmark_index"
    )

    for i in range(HAND_COUNT):

        row = left_df[
            left_df["landmark_index"] == i
        ]

        if len(row) > 0:

            r = row.iloc[0]

            values.extend([
                r["x"],
                r["y"],
                r["z"]
            ])

        else:

            values.extend([
                np.nan,
                np.nan,
                np.nan
            ])

    # --------------------------------------------------------
    # Right hand
    # --------------------------------------------------------

    right_df = frame_df[
        frame_df["type"] == "right_hand"
    ].sort_values(
        "landmark_index"
    )

    for i in range(HAND_COUNT):

        row = right_df[
            right_df["landmark_index"] == i
        ]

        if len(row) > 0:

            r = row.iloc[0]

            values.extend([
                r["x"],
                r["y"],
                r["z"]
            ])

        else:

            values.extend([
                np.nan,
                np.nan,
                np.nan
            ])

    # --------------------------------------------------------
    # Verify
    # --------------------------------------------------------

    if len(values) != 225:

        raise RuntimeError(
            f"Frame {frame_id}: "
            f"Expected 225 features, "
            f"got {len(values)}"
        )

    frame_features.append(values)

X = np.asarray(
    frame_features,
    dtype=np.float32
)

print(
    "\nRaw reconstructed shape:",
    X.shape
)

# ============================================================
# 8. CONVERT TO EXACTLY 40 FRAMES
# ============================================================

if X.shape[0] >= 40:

    indices = np.linspace(
        0,
        X.shape[0] - 1,
        40
    ).astype(int)

    X = X[indices]

else:

    padding_count = (
        40 - X.shape[0]
    )

    padding = np.repeat(
        X[-1][None, :],
        padding_count,
        axis=0
    )

    X = np.concatenate(
        [
            X,
            padding
        ],
        axis=0
    )

print(
    "40-frame shape:",
    X.shape
)

# ============================================================
# 9. HANDLE MISSING VALUES
# ============================================================

X = np.nan_to_num(
    X,
    nan=0.0,
    posinf=0.0,
    neginf=0.0
)

# ============================================================
# 10. NORMALIZE
# ============================================================

X = (
    X - mean
) / std

X = X.astype(
    np.float32
)

print(
    "Normalized shape:",
    X.shape
)

# ============================================================
# 11. ADD BATCH DIMENSION
# ============================================================

X_input = X[
    np.newaxis,
    :,
    :
]

print(
    "FINAL MODEL INPUT:",
    X_input.shape
)

if X_input.shape != (
    1,
    40,
    225
):

    raise RuntimeError(
        f"Wrong shape: {X_input.shape}"
    )

# ============================================================
# 12. PREDICT
# ============================================================

print("\n" + "=" * 65)
print("RUNNING MODEL")
print("=" * 65)

probabilities = model.predict(
    X_input,
    verbose=0
)[0]

# ============================================================
# 13. TOP 10
# ============================================================

top_indices = np.argsort(
    probabilities
)[::-1][:10]

print("\n" + "=" * 65)
print("TOP 10 PREDICTIONS")
print("=" * 65)

for rank, idx in enumerate(
    top_indices,
    start=1
):

    print(
        f"{rank:2d}. "
        f"{classes[idx]:25s} "
        f"{probabilities[idx] * 100:7.2f}%"
    )

# ============================================================
# 14. FINAL RESULT
# ============================================================

best_idx = top_indices[0]

predicted_label = classes[
    best_idx
]

true_label = "cat"

print("\n" + "=" * 65)
print("FINAL RESULT")
print("=" * 65)

print(
    "True label      :",
    true_label
)

print(
    "Prediction      :",
    predicted_label
)

print(
    f"Confidence      : "
    f"{probabilities[best_idx] * 100:.2f}%"
)

if predicted_label == true_label:

    print(
        "\nRESULT          : ✅ CORRECT"
    )

else:

    print(
        "\nRESULT          : ❌ INCORRECT"
    )

# ============================================================
# 15. TRUE CLASS RANK
# ============================================================

true_idx = classes.index(
    true_label
)

true_probability = (
    probabilities[true_idx] * 100
)

true_rank = (
    np.where(
        np.argsort(probabilities)[::-1]
        == true_idx
    )[0][0] + 1
)

print(
    "\nTrue class probability:",
    f"{true_probability:.2f}%"
)

print(
    "True class rank:",
    true_rank
)

print("=" * 65)

CHECKING FILES
✓ /kaggle/working/signora_model/best_isl_bigru.keras
✓ /kaggle/working/signora_model/classes.json
✓ /kaggle/working/signora_model/mean.npy
✓ /kaggle/working/signora_model/std.npy
✓ /kaggle/working/keypoints/animals/cat/MVI_2982.parquet

Model input : (None, 40, 225)
Model output: (None, 262)
Classes: 262

Original mean shape: (1, 1, 225)
Original std shape : (1, 1, 225)
Mean shape: (225,)
Std shape : (225,)

LOADING ORIGINAL TRAINING LANDMARKS
Parquet shape: (30415, 7)

Columns:
['frame', 'row_id', 'type', 'landmark_index', 'x', 'y', 'z']

Landmark types:
type
face          26290
pose           1815
left_hand      1155
right_hand     1155
Name: count, dtype: int64

Number of frames: 55

Frames found: 55

Raw reconstructed shape: (55, 225)
40-frame shape: (40, 225)
Normalized shape: (40, 225)
FINAL MODEL INPUT: (1, 40, 225)

RUNNING MODEL

TOP 10 PREDICTIONS
 1. cat                         58.17%
 2. truck                       30.47%
 3. transportation               6.49